In [1]:
from jira import JIRA

from datetime import datetime,timedelta
import sys
sys.path.append('E:\wangzhilin\QuantumGalaxy')
from QGI.mysql import MYSQL
import numpy as np
import pandas as pd
import json

In [2]:
import logging
def get_logger():
    logger = logging.getLogger('logger')
    logger.setLevel(logging.INFO)
    #fh = logging.FileHandler(
    #    "E:/wangzhilin/QuantumGalaxy/logs/flask_server.log",
    #    'a',
    #    encoding='utf-8')
    #fh.setLevel(logging.INFO)
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    formatter = logging.Formatter(
        fmt='%(asctime)s %(name)-12s %(levelname)-8s %(message)s',
        datefmt='%m-%d %H:%M')
    ch.setFormatter(formatter)
    #fh.setFormatter(formatter)
    logger.addHandler(ch)
    #logger.addHandler(fh)
    return logger
logger=get_logger()

In [3]:
code='688063.SH'


In [4]:
import scipy,math
def log_inter1(num:int,y0,y1):
    '''返回长度为num+2的插值列表。num:插值数量'''
    f=scipy.interpolate.interp1d(x=[0,num+1],y=[math.log(y0),math.log(y1)],kind='linear')
    res=f([x+1 for x in range(num)])

    res=[math.exp(r) for r in res]
    res.insert(0,y0)
    res.append(y1)
    return res
def geo_mean(data):
    return math.pow(np.prod(data),1.0/len(data))
def calcu_norm(xc,pc,xo,po,ppf) :
    '''-> (s,mu)'''
    xc=np.log(xc)
    xo=np.log(xo)
    no=round((1-po)*10-1)
    nc=round((1-pc)*10-1)
    #print(no,nc)
    s=(xo-xc)/(ppf[no]-ppf[nc])
    mu=((xo-ppf[no]*s+xc-ppf[nc]*s)/2)
    return(s,mu)

In [5]:
xo=issue.fields.customfield_10918
po=issue.fields.customfield_10919
xc=issue.fields.customfield_10916
pc=issue.fields.customfield_10914
ppf = [-1.282, -0.842, -0.524, -0.253, 0, 0.253, 0.524, 0.842, 1.282]
calcu_norm(xc=xc,xo=xo,pc=pc,po=po,ppf=ppf)

NameError: name 'issue' is not defined

In [6]:
def data_process(code):
    host='localhost'
    user='Local_Editor'
    password='QuantumGalaxy'
    database='qgdbs'
    mysql=MYSQL(host,user,password,database)
    jira= JIRA('https://research.quantumgalaxy.cn/', basic_auth=('wangzhilin', "wangzhilin"))
    try:
        issue=jira.search_issues(f'project = POSITION AND status != 尚未纳入 AND resolution = Unresolved AND cf[10201] ~  {code}')[0]
    except:
        logger.error(f'no position issue, code: {code}')
        return False
    target_date=issue.fields.customfield_10622
    start_date=issue.fields.updated[:10]

    xo=issue.fields.customfield_10918
    po=issue.fields.customfield_10919
    xc=issue.fields.customfield_10916
    pc=issue.fields.customfield_10914
    
    stock_start_date=datetime.strptime(start_date,'%Y-%m-%d')-timedelta(183)
    stock_start_date=stock_start_date.strftime('%Y-%m-%d')


    ev_data=mysql.read_query('select date,market_value from ticker_data where code="%s" and date<=now() and date>="%s"'%(code,stock_start_date))
    ev_data=[ev for ev in ev_data]
    stock_end_date=ev_data[-1][0]

    import math


    ppf = [-1.282, -0.842, -0.524, -0.253, 0, 0.253, 0.524, 0.842, 1.282]


    s,mu=calcu_norm(xc=xc,xo=xo,pc=pc,po=po,ppf=ppf)
    #print(s,mu)
    df=pd.DataFrame(data=ev_data,columns=['date','ev'])
    df.set_index('date',inplace=True)


    stock_data=mysql.read_query(f'select date,close_price from ticker_data where code="{code}" and date<="{start_date}" order by date desc limit 60')
    vol=mysql.read_query(f'select vol from processed_data where code="{code}" and date<="{start_date}" order by date desc limit 1')
    vol=vol[0][0]
    stock_data=[float(p) for (d,p) in stock_data]
    stock_data=geo_mean(stock_data)

    factor=[(1-vol)**10,(1-vol)**5,1,(1+vol)**5,(1+vol)**10,]
    start=np.outer(stock_data,factor)[0]
    n=mysql.read_query('select value from customized_data where code="C00002.QG" order by date desc limit 1')[0][0]
    end=np.exp([mu-0.842*s,mu-0.5*s,mu,mu+0.5*s,mu+0.842*s])*n

    start_date=datetime.strptime(start_date,'%Y-%m-%d')
    target_date=datetime.strptime(target_date,'%Y-%m-%d')
    days=(target_date-start_date).days
    interp_num=days-1
    day_list=[start_date+timedelta(i) for i in range(days+1)]
    day_list=[datetime.date(d) for d in day_list]
    level0=log_inter1(interp_num,start[0],end[0])

    level1=log_inter1(interp_num,start[1],end[1])

    level2=log_inter1(interp_num,start[2],end[2])

    level3=log_inter1(interp_num,start[3],end[3])

    level4=log_inter1(interp_num,start[4],end[4])
    level4=[level4[i]-level3[i] for i in range(len(level4))]
    level3=[level3[i]-level2[i] for i in range(len(level4))]
    level2=[level2[i]-level1[i] for i in range(len(level4))]
    level1=[level1[i]-level0[i] for i in range(len(level4))]
    level0=tuple(dict(zip(day_list,level0)).items())
    level1=tuple(dict(zip(day_list,level1)).items())
    level2=tuple(dict(zip(day_list,level2)).items())
    level3=tuple(dict(zip(day_list,level3)).items())
    level4=tuple(dict(zip(day_list,level4)).items())
    down_pred_point=(level0[-1][0],level0[-1][1]+level1[-1][1])
    pred_point=(level0[-1][0],level0[-1][1]+level1[-1][1]+level2[-1][1])
    up_pred_point=(level0[-1][0],level0[-1][1]+level1[-1][1]+level2[-1][1]+level3[-1][1])
    down_pred_point=np.round(down_pred_point[1],2)
    pred_point=np.round(pred_point[1],2)

    up_pred_point=np.round(up_pred_point[1],2)
    df['last_ev']=None
    df['last_ev'][-1]=df['ev'][-1]
    df1=pd.DataFrame(level0,columns=['date','level0'])
    df1.set_index('date',inplace=True)
    df=pd.concat([df,df1],axis=1)
    df1=pd.DataFrame(level1,columns=['date','level1'])
    df1.set_index('date',inplace=True)
    df=pd.concat([df,df1],axis=1)
    df1=pd.DataFrame(level2,columns=['date','level2'])
    df1.set_index('date',inplace=True)
    df=pd.concat([df,df1],axis=1)
    df1=pd.DataFrame(level3,columns=['date','level3'])
    df1.set_index('date',inplace=True)
    df=pd.concat([df,df1],axis=1)
    df1=pd.DataFrame(level4,columns=['date','level4'])
    df1.set_index('date',inplace=True)
    df=pd.concat([df,df1],axis=1)
    df['down_pred_point']=None
    df['pred_point']=None
    df['up_pred_point']=None
    down_pred_point=(level0[-1][0],level0[-1][1]+level1[-1][1])
    pred_point=(level0[-1][0],level0[-1][1]+level1[-1][1]+level2[-1][1])
    up_pred_point=(level0[-1][0],level0[-1][1]+level1[-1][1]+level2[-1][1]+level3[-1][1])
    down_pred_point=np.round(down_pred_point[1],2)
    pred_point=np.round(pred_point[1],2)

    up_pred_point=np.round(up_pred_point[1],2)
    
    df.index=df.index.astype('str',)
    df=df.reset_index()
    df.loc[df.shape[0]-1,'down_pred_point']=down_pred_point
    df.loc[df.shape[0]-1,'up_pred_point']=up_pred_point
    df.loc[df.shape[0]-1,'pred_point']=pred_point
    df = df.astype(object).replace(np.nan, 'None')
    dataset={}
    dataset['source']=[df.columns.to_list()]
    dataset['source']
    for i,r in df.iterrows():

        row=r.to_list()
        dataset['source'].append(row)
    j=json.dumps(dataset)
    return j

In [10]:
issue.fields.__dict__

{'issuetype': <JIRA IssueType: name='交易任务', id='10402'>,
 'timespent': None,
 'project': <JIRA Project: key='POSITION', name='持仓交易管理项目', id='10302'>,
 'customfield_11441': '2.89, 1.59',
 'customfield_11002': None,
 'aggregatetimespent': None,
 'resolution': None,
 'customfield_11003': None,
 'customfield_11720': None,
 'customfield_10631': None,
 'customfield_11400': 11.161655546987491,
 'customfield_10302': None,
 'customfield_10621': None,
 'customfield_10104': ['com.atlassian.greenhopper.service.sprint.Sprint@de9697d[id=9,rapidViewId=9,state=ACTIVE,name=高度观察+持仓列表,startDate=2022-03-06T01:24:00.000+08:00,endDate=2023-12-31T01:24:00.000+08:00,completeDate=<null>,activatedDate=2021-03-06T01:24:16.619+08:00,sequence=9,goal=<null>,autoStartStop=false]'],
 'customfield_10622': '2023-04-30',
 'customfield_10105': '1|i00idr:',
 'customfield_10700': None,
 'customfield_11438': 0.5134361445168404,
 'customfield_10701': None,
 'customfield_10625': None,
 'customfield_10702': None,
 'customfield

In [9]:
jira= JIRA('https://research.quantumgalaxy.cn/', basic_auth=('wangzhilin', "wangzhilin"))
try:
    issue=jira.search_issues(f'project = POSITION AND status != 尚未纳入 AND resolution = Unresolved AND cf[10201] ~  {code}')[0]
except:
    logger.error(f'no position issue, code: {code}')

target_date=issue.fields.customfield_10622
start_date=issue.fields.updated[:10]

xo=issue.fields.customfield_10918
po=issue.fields.customfield_10919
xc=issue.fields.customfield_10916
pc=issue.fields.customfield_10914

In [12]:
from atlassian import Jira

jira = Jira(
    url='https://research.quantumgalaxy.cn/',
    username='wangzhilin',
    password='wangzhilin')


In [16]:
issue=jira.jql(f'project = POSITION AND status != 尚未纳入 AND resolution = Unresolved AND cf[10201] ~  {code}')

In [26]:
issue['issues'][0]['key']

'POSITION-2969'

In [32]:
log=jira.get_issue_changelog('POSITION-2969')['histories']

In [33]:
log

[{'id': '114269',
  'author': {'self': 'https://research.quantumgalaxy.cn/rest/api/2/user?username=bot',
   'name': 'bot',
   'key': 'JIRAUSER10200',
   'emailAddress': 'jira_bot@quantumgalaxy.cn',
   'avatarUrls': {'48x48': 'https://www.gravatar.com/avatar/3d75f49b419a66aa39f16798646dd968?d=mm&s=48',
    '24x24': 'https://www.gravatar.com/avatar/3d75f49b419a66aa39f16798646dd968?d=mm&s=24',
    '16x16': 'https://www.gravatar.com/avatar/3d75f49b419a66aa39f16798646dd968?d=mm&s=16',
    '32x32': 'https://www.gravatar.com/avatar/3d75f49b419a66aa39f16798646dd968?d=mm&s=32'},
   'displayName': '研究机器人',
   'active': True,
   'timeZone': 'Asia/Shanghai'},
  'created': '2021-10-04T12:45:44.000+0800',
  'items': [{'field': '万得代码',
    'fieldtype': 'custom',
    'from': None,
    'fromString': None,
    'to': None,
    'toString': '688063.SH'}]},
 {'id': '114270',
  'author': {'self': 'https://research.quantumgalaxy.cn/rest/api/2/user?username=bot',
   'name': 'bot',
   'key': 'JIRAUSER10200',
  

In [35]:
log[1]

{'id': '114270',
 'author': {'self': 'https://research.quantumgalaxy.cn/rest/api/2/user?username=bot',
  'name': 'bot',
  'key': 'JIRAUSER10200',
  'emailAddress': 'jira_bot@quantumgalaxy.cn',
  'avatarUrls': {'48x48': 'https://www.gravatar.com/avatar/3d75f49b419a66aa39f16798646dd968?d=mm&s=48',
   '24x24': 'https://www.gravatar.com/avatar/3d75f49b419a66aa39f16798646dd968?d=mm&s=24',
   '16x16': 'https://www.gravatar.com/avatar/3d75f49b419a66aa39f16798646dd968?d=mm&s=16',
   '32x32': 'https://www.gravatar.com/avatar/3d75f49b419a66aa39f16798646dd968?d=mm&s=32'},
  'displayName': '研究机器人',
  'active': True,
  'timeZone': 'Asia/Shanghai'},
 'created': '2021-10-04T12:45:45.000+0800',
 'items': [{'field': 'Sprint',
   'fieldtype': 'custom',
   'from': None,
   'fromString': None,
   'to': '9',
   'toString': '入选的持仓组合'}]}

In [141]:
df=data_process(code)


10-10 03:54 logger       INFO     using username Local_Editor
10-10 03:54 logger       INFO     using username Local_Editor
10-10 03:54 logger       INFO     using username Local_Editor
10-10 03:54 logger       INFO     accessed to database <qgdbs>
10-10 03:54 logger       INFO     accessed to database <qgdbs>
10-10 03:54 logger       INFO     accessed to database <qgdbs>
10-10 03:54 logger       INFO     got a cursor
10-10 03:54 logger       INFO     got a cursor
10-10 03:54 logger       INFO     got a cursor
10-10 03:54 logger       INFO     Accepted read sql query "select date,market_value from ticker_data where code="688063.SH" and date<=now() and date>="2022-04-09""
10-10 03:54 logger       INFO     Accepted read sql query "select date,market_value from ticker_data where code="688063.SH" and date<=now() and date>="2022-04-09""
10-10 03:54 logger       INFO     Accepted read sql query "select date,market_value from ticker_data where code="688063.SH" and date<=now() and date>="2022-

6 3


C:\Users\wangzhilin\AppData\Local\Temp\2\ipykernel_51256\2839177745.py:84: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['last_ev'][-1]=df['ev'][-1]


In [142]:

df

,date,ev,last_ev,level0,level1,level2,level3,level4,down_pred_point,pred_point,up_pred_point
0,2022-04-11,196.652557,None,None,None,None,None,None,None,None,None
1,2022-04-12,213.685456,None,None,None,None,None,None,None,None,None
2,2022-04-13,207.770394,None,None,None,None,None,None,None,None,None
3,2022-04-14,216.689439,None,None,None,None,None,None,None,None,None
4,2022-04-15,229.711865,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...
323,2023-04-26,None,None,197.938479,141.391398,405.269204,888.944293,1166.277964,None,None,None
324,2023-04-27,None,None,197.68057,141.720926,407.409636,896.192968,1177.451383,None,None,None
325,2023-04-28,None,None,197.422996,142.050134,409.556624,903.489864,1188.722079,None,None,None
326,2023-04-29,None,None,197.165758,142.379021,411.710188,910.835278,1200.090856,None,None,None
